In [11]:
import pandas as pd
import os
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

In [2]:
# load pandas parquet from /var/db/categorizedsummaries/Other
df = pd.read_parquet("/var/db/categorizedsummaries/Other")

In [3]:
df.columns

Index(['commit_hash', 'category', 'semantic_summary', 'repo_id',
       'commit_message', 'embedding'],
      dtype='str')

In [4]:
# unique categories
df["category"].unique()

<ArrowStringArray>
['Other']
Length: 1, dtype: str

In [1]:
import app.vectordb.categorize_commits as cc
from app.vectordb.chroma_config import ChromaConfig



In [2]:
config = ChromaConfig(collection_name="test1")
ids, embeddings, metadatas, documents = cc.fetch_embeddings(config)
batch_labels = cc.batch_summaries_kmeans(embeddings, 20)
prompt_text = cc.load_prompt(cc.CATEGORY_DISCOVERY_PROMPT_PATH)
chain = cc.llm_chain(prompt_text)
summaries = documents

In [3]:
batch = batch_labels[0]

In [ ]:
batch_summaries = [summaries[i] for i in range(len(summaries)) if batch_labels[i] == batch]
input_text = prompt_text + "\n" + "\n".join([f"Summary {i+1}: {s}" for i, s in enumerate(batch_summaries)])
llm = ChatOpenAI(temperature=0, model_name=os.getenv("LMSTUDIO_MODEL", "gpt-oss-20b"), openai_api_base=os.getenv(
    "LMSTUDIO_BASE_URL", "http://host.docker.internal:1234/v1"), openai_api_key=os.getenv("LMSTUDIO_API_KEY", "not-needed"))

result = llm.invoke(input=input_text)

In [10]:
result

AIMessage(content='- Rendering Improvements\n- Bug Fixes\n- Feature Additions\n- Error Handling\n- Compatibility Support\n- Documentation Updates\n- Mod Integration\n- Inventory Management\n- AI Enhancements\n- Visual Customization\n- Gameplay Mechanics\n- User Experience Improvements\n- Recipe Information\n- Experience Rewards\n- Customization Options', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 2519, 'total_tokens': 2586, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'llama-3.1-8b-instruct', 'system_fingerprint': 'llama-3.1-8b-instruct', 'id': 'chatcmpl-pnbx3zbxb3ovo6kvco7lsi', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd3de-5f80-7182-82c3-cbab9a1a5cf5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [13]:
class CategoryList(BaseModel):
    categories: list[str]

In [14]:
llm_w_struct = llm.with_structured_output(CategoryList)

In [16]:
cat_list: CategoryList = llm_w_struct.invoke(input=input_text)

In [17]:
cat_list.model_dump()

{'categories': ['Compatibility Fixes',
  'Feature Additions',
  'Gameplay Mechanics',
  'Inventory Management',
  'Mod Integration',
  'Performance Optimizations',
  'Recipe Enhancements',
  'User Interface Updates']}